# Market Regime Classification: Filter Your Strategies (Without Fooling Yourself)

A market regime is a period in which the statistical behaviour of the market is relatively stable. Every strategy is implicitly a bet on a regime: a trend follower needs trends, a mean reversion system needs ranges. This notebook measures regimes along two dimensions (**directionality** and **volatility**), classifies them with rules and with a Hidden Markov Model, and uses them to filter two strategies with opposite preferences. Most of the work is about checking that the filter adds value, not just that it makes the backtest prettier. All backtests are demonstrations (one market, daily bars), not validations.

**Contents**

1. Data and two strategies with opposite regime preferences
2. An asymmetry: volatility is persistent, direction is not
3. Indicators of volatility and directionality, with rolling percentile thresholds
4. Classification: rule-based 2×2 matrix and Hidden Markov Model (filtered probabilities)
5. Trap #1: look-ahead bias
6. Persistence: transition matrix, regime duration and indicator lag
7. Trap #2: whipsaw at the thresholds
8. Strategy performance in the four quadrants
9. Binary filter vs position sizing
10. Trade-level analysis: scatter, correlations, bins, 2D heatmap
11. Beyond the final result: MAE and MFE by regime
12. Trap #3: data snooping
13. Validation: thresholds in-sample, check out-of-sample, plateau vs peak
14. Anchored walk-forward on the filter, and threshold stability
15. The random filter benchmark and the final comparison
16. The holdout: one run, at the end

## 0. Setup and data

Install the dependencies with `pip install numpy pandas scipy scikit-learn statsmodels hmmlearn yfinance plotly`.

The two blocks of parameters below are the only knobs of the notebook. Regimes always use **relative** values (rolling percentiles), the strategy parameters are standard and fixed, and the last years of data (`HOLDOUT_START`) are cut away right after the download: nothing in sections 1–15 touches them, not even the choice of indicators.

In [1]:
import warnings
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import stats
from hmmlearn.hmm import GaussianHMM
from sklearn.feature_selection import mutual_info_regression
from statsmodels.nonparametric.smoothers_lowess import lowess
from statsmodels.tsa.stattools import acf

warnings.filterwarnings("ignore")
pio.templates.default = "plotly_dark"
# if charts don't show up (some VS Code / JupyterLab versions): pio.renderers.default = "notebook"

In [2]:
# --- market and sample split ---
TICKER = "^GSPC"            # any liquid index/instrument works: "^NDX", "^GDAXI", "GC=F", ...
START = "2005-01-01"
END = "2026-10-01"          # fixed end date, so the notebook always reproduces the same numbers
IS_END = "2014-12-31"       # in-sample: up to here; out-of-sample: until HOLDOUT_START
HOLDOUT_START = "2023-01-01"  # never used before section 16

# --- regime definition (all thresholds are rolling percentiles) ---
PCT_WINDOW = 252            # window of the rolling percentile (1 year)
VOL_Q = 0.70                # high volatility = realized vol above its rolling 70th percentile
DIR_Q = 0.55                # trending = efficiency ratio above its rolling 55th percentile
BAND = 0.10                 # hysteresis: exit threshold = entry threshold -/+ BAND
MIN_DUR = 5                 # minimum duration (bars) to confirm a regime change

# --- strategies (fixed, standard, not optimized) ---
ATR_N, ATR_K = 14, 3.0      # initial stop at ATR_K * ATR: it defines the risk (1R) of every trade
COST = 0.0002               # 2 bps per side
BARS_PER_YEAR = 252

In [3]:
data_all = yf.download(TICKER, start=START, end=END, auto_adjust=True, progress=False)
data_all.columns = data_all.columns.get_level_values(0)
data_all = data_all[["Open", "High", "Low", "Close"]].dropna()

prices = data_all[data_all.index < HOLDOUT_START]       # development data: sections 1-15
print(f"{TICKER}: {len(prices)} daily bars, {prices.index[0].date()} -> {prices.index[-1].date()}")
print(f"IS until {IS_END}, OOS until {prices.index[-1].date()}, holdout from {HOLDOUT_START} (locked)")

^GSPC: 4531 daily bars, 2005-01-03 -> 2022-12-30
IS until 2014-12-31, OOS until 2022-12-30, holdout from 2023-01-01 (locked)


In [4]:
def runs(mask):
    """Start/end dates of the consecutive True runs of a boolean series."""
    m = mask.fillna(False).astype(bool)
    start = m & ~m.shift(1, fill_value=False)
    end = m & ~m.shift(-1, fill_value=False)
    return list(zip(m.index[start], m.index[end]))

def shade(fig, mask, color, row=None, col=None, opacity=0.25):
    for a, b in runs(mask):
        fig.add_vrect(x0=a, x1=b, fillcolor=color, opacity=opacity, line_width=0, layer="below", row=row, col=col)

def plot_lines(series, title, ytitle=None, logy=False, height=550, shades=(), mode="lines"):
    """Time-series helper: dict name -> Series, optional shaded regimes [(mask, color), ...]."""
    fig = go.Figure([go.Scatter(x=s.index, y=s, name=n, mode=mode, line_width=2) for n, s in series.items()])
    for mask, color in shades:
        shade(fig, mask, color)
    fig.update_layout(title=title, yaxis_title=ytitle, height=height, hovermode="x unified")
    if logy:
        fig.update_yaxes(type="log")
    fig.show()

def sharpe(r):
    return r.mean() / r.std() * np.sqrt(BARS_PER_YEAR) if r.std() > 0 else np.nan

def max_dd(r):
    eq = (1 + r).cumprod()
    return (eq / eq.cummax() - 1).min()

def equity(r):
    return (1 + r).cumprod()

## 1. Data and two strategies with opposite regime preferences

Two long-only strategies on the index, with simple textbook rules and fixed parameters (nothing here is optimized):

- **Trend follower**: buy when the close breaks the 20-day high, exit when the close breaks the 10-day low.
- **Mean reversion**: buy when RSI(2) drops below 10 (a very short-term oversold reading), exit when the close goes back above its 5-day average.

Common rules: the signal is evaluated at the **close** of bar *t*, the trade is entered at the **open** of *t+1*, and an exit signal on a close is executed at the next open. Every trade has an initial stop at 3 × ATR(14), which defines the initial risk: the result of each trade is expressed in **R-multiples** (profit divided by initial risk), so trades taken in calm and in turbulent markets are comparable. Costs are 2 bps per side.

In [5]:
def atr(df, n=ATR_N):
    pc = df.Close.shift(1)
    tr = pd.concat([df.High - df.Low, (df.High - pc).abs(), (df.Low - pc).abs()], axis=1).max(axis=1)
    return tr.ewm(alpha=1 / n, adjust=False, min_periods=n).mean()

def rsi(c, n=2):
    d = c.diff()
    up = d.clip(lower=0).ewm(alpha=1 / n, adjust=False, min_periods=n).mean()
    dn = (-d.clip(upper=0)).ewm(alpha=1 / n, adjust=False, min_periods=n).mean()
    return 100 - 100 / (1 + up / dn)

def backtest(df, entry, exit_, name):
    """Long-only event backtest. Returns the trades table (R, MAE, MFE in units of initial risk),
    the per-bar returns of the strategy and the id of the trade that is open on each bar."""
    o, h, l, c = (df[k].to_numpy() for k in ["Open", "High", "Low", "Close"])
    a, en, ex = atr(df).to_numpy(), entry.to_numpy(), exit_.to_numpy()
    n = len(df)
    ret, tid = np.zeros(n), np.full(n, -1)
    trades, i = [], 0
    while i < n - 1:
        if not (en[i] and not np.isnan(a[i])):
            i += 1
            continue
        k, j0 = len(trades), i + 1                     # signal on bar i, entry at the open of bar i+1
        px = o[j0]
        stop = px - ATR_K * a[i]
        risk = px - stop
        prev, mae, mfe, j = px, 0.0, 0.0, j0
        while True:
            mae, mfe = max(mae, px - l[j]), max(mfe, h[j] - px)
            tid[j] = k
            if l[j] <= stop:                           # stop hit (gap: filled at the open)
                xpx = min(o[j], stop) if j > j0 else stop
                ret[j] += xpx / prev - 1 - COST - (COST if j == j0 else 0)
                break
            ret[j] += c[j] / prev - 1 - (COST if j == j0 else 0)
            prev = c[j]
            if j == n - 1:                             # end of data: mark to market
                xpx = c[j]
                break
            if ex[j]:                                  # exit signal on the close -> exit at the next open
                j += 1
                xpx = o[j]
                tid[j] = k
                ret[j] += xpx / prev - 1 - COST
                break
            j += 1
        trades.append(dict(strategy=name, entry_date=df.index[j0], exit_date=df.index[j], bars=j - j0 + 1,
                           R=(xpx - px) / risk - 2 * COST * px / risk, MAE=mae / risk, MFE=mfe / risk))
        i = j
    return pd.DataFrame(trades), pd.Series(ret, df.index), pd.Series(tid, df.index)

def strategies(df):
    c, h, l = df.Close, df.High, df.Low
    trend = backtest(df, c > h.rolling(20).max().shift(1), c < l.rolling(10).min().shift(1), "trend")
    meanrev = backtest(df, rsi(c, 2) < 10, c > c.rolling(5).mean(), "meanrev")
    return {"trend": trend, "meanrev": meanrev}

In [6]:
raw_strats = strategies(prices)
bh = prices.Close.pct_change().fillna(0)

rows = {}
for name, (tr, r, tid) in raw_strats.items():
    rows[name] = dict(trades=len(tr), win_rate=(tr.R > 0).mean(), expectancy_R=tr.R.mean(),
                      avg_bars=tr.bars.mean(), time_in_market=(tid >= 0).mean(), sharpe=sharpe(r), max_dd=max_dd(r))
rows["buy & hold"] = dict(sharpe=sharpe(bh), max_dd=max_dd(bh), time_in_market=1.0)
display(pd.DataFrame(rows).T.round(2))

plot_lines({"buy & hold": equity(bh), "trend follower": equity(raw_strats["trend"][1]),
            "mean reversion": equity(raw_strats["meanrev"][1])},
           f"Equity curves, fully invested while in a trade ({TICKER}, log scale)", logy=True)

,trades,win_rate,expectancy_R,avg_bars,time_in_market,sharpe,max_dd
trend,80.0,0.50,0.21,30.70,0.54,0.45,-0.27
meanrev,224.0,0.68,0.08,4.08,0.20,0.34,-0.30
buy & hold,NaN,NaN,NaN,NaN,1.00,0.43,-0.57


Over the whole development sample the trend follower has a Sharpe ratio of 0.45 (80 trades, 50% win rate, expectancy 0.21R, in the market 54% of the time) and the mean reversion 0.34 (224 trades, 68% win rate, 0.08R, in the market 20% of the time); buy & hold has 0.43. Their maximum drawdowns (-27% and -30%) are about half of the -57% of the index. These are modest edges, and they are not constant: the yearly expectancy changes a lot from one year to the next (for example the trend follower loses 0.41R per trade on average in 2008 and earns 1.18R in 2017), which is exactly the question of this notebook: can we tell, *using only the past*, when to trade?

In [7]:
yearly = pd.DataFrame({n: tr.groupby(tr.entry_date.dt.year).R.mean() for n, (tr, _, _) in raw_strats.items()})
fig = go.Figure([go.Bar(x=yearly.index, y=yearly[c], name=c) for c in yearly])
fig.update_layout(title="Average R-multiple per trade, by year of entry", yaxis_title="R", barmode="group", height=450)
fig.show()

## 2. An asymmetry: volatility is persistent, direction is not

Before measuring anything, one fact shapes the whole notebook. If today's volatility says a lot about tomorrow's, a volatility regime can be recognised and used. If today's direction says little about tomorrow's, directional regimes are much harder to exploit. The autocorrelation function (ACF) shows it: the ACF of the returns against the ACF of the **absolute** returns (a simple proxy of volatility). The dashed lines are the 95% bands of white noise.

In [8]:
logret = np.log(prices.Close).diff().dropna()
NLAGS = 40
band = 1.96 / np.sqrt(len(logret))
acf_r, acf_abs = acf(logret, nlags=NLAGS)[1:], acf(logret.abs(), nlags=NLAGS)[1:]

fig = make_subplots(rows=1, cols=2, subplot_titles=("ACF of returns", "ACF of absolute returns"), shared_yaxes=True)
for col, v in enumerate([acf_r, acf_abs], 1):
    fig.add_trace(go.Bar(x=np.arange(1, NLAGS + 1), y=v, showlegend=False), row=1, col=col)
    for s in (band, -band):
        fig.add_hline(y=s, line_dash="dash", line_color="white", row=1, col=col)
    fig.update_xaxes(title_text="lag (days)", row=1, col=col)
fig.update_layout(height=450, title="Returns are almost uncorrelated, volatility is persistent")
fig.show()

print(f"95% band: ±{band:.3f}")
print(f"returns:          lag 1 = {acf_r[0]:+.3f}, mean of lags 1-10 = {acf_r[:10].mean():+.3f}, lags outside the band: {(np.abs(acf_r) > band).sum()}/{NLAGS}")
print(f"absolute returns: lag 1 = {acf_abs[0]:+.3f}, lag 20 = {acf_abs[19]:+.3f}, lag 40 = {acf_abs[39]:+.3f}, lags outside the band: {(np.abs(acf_abs) > band).sum()}/{NLAGS}")

95% band: ±0.029
returns:          lag 1 = -0.129, mean of lags 1-10 = -0.013, lags outside the band: 13/40
absolute returns: lag 1 = +0.327, lag 20 = +0.256, lag 40 = +0.174, lags outside the band: 40/40


The ACF of the absolute returns is positive and significant at **all** 40 lags: +0.33 at lag 1, +0.26 at lag 20, +0.17 at lag 40. Volatility has a long memory (volatility clustering), so today's volatility tells us a lot about the next weeks. The ACF of the returns is small: the mean of the first 10 lags is -0.01. There is a significant negative autocorrelation at lag 1 (-0.13) and 13 of 40 lags are outside the band, but the values are tiny compared with the volatility case. In short, direction is only weakly persistent and hard to predict: keep this asymmetry in mind when we read the results of the volatility filters and of the directional ones.

## 3. Indicators of volatility and directionality

Two families, two indicators each plus one statistical measure (all computed on daily bars):

**Volatility**
- **Realized volatility** (`rv`): annualized rolling standard deviation of the last 20 log returns.
- **ATR %** (`atrp`): ATR(14) divided by the close.

**Directionality**
- **Efficiency Ratio** (`er`, Kaufman): net price change over 20 bars divided by the sum of the absolute daily changes. A straight line gives 1, a zigzag that ends where it started gives 0.
- **ADX** (`adx`): Wilder's trend strength, 14 bars.
- **Variance ratio** (`vr`, Lo–MacKinlay): variance of 5-day returns divided by 5 times the variance of daily returns, over 120 bars. Above 1 the returns are positively autocorrelated (trending), below 1 they mean-revert.

In [9]:
def efficiency_ratio(c, n=20):
    return (c - c.shift(n)).abs() / c.diff().abs().rolling(n).sum()

def adx(df, n=14):
    up, dn = df.High.diff(), -df.Low.diff()
    pdm = pd.Series(np.where((up > dn) & (up > 0), up, 0.0), df.index)
    mdm = pd.Series(np.where((dn > up) & (dn > 0), dn, 0.0), df.index)
    a = atr(df, n)
    pdi = 100 * pdm.ewm(alpha=1 / n, adjust=False, min_periods=n).mean() / a
    mdi = 100 * mdm.ewm(alpha=1 / n, adjust=False, min_periods=n).mean() / a
    dx = 100 * (pdi - mdi).abs() / (pdi + mdi)
    return dx.ewm(alpha=1 / n, adjust=False, min_periods=n).mean()

def variance_ratio(r, q=5, n=120):
    return r.rolling(q).sum().rolling(n).var() / (q * r.rolling(n).var())

def indicators(df):
    r = np.log(df.Close).diff()
    return pd.DataFrame({"rv": r.rolling(20).std() * np.sqrt(BARS_PER_YEAR), "atrp": atr(df) / df.Close,
                         "er": efficiency_ratio(df.Close), "adx": adx(df), "vr": variance_ratio(r)}, index=df.index)

VOL_IND, DIR_IND = ["rv", "atrp"], ["er", "adx", "vr"]
ind = indicators(prices)
display(ind.describe().T.round(3))

,count,mean,std,min,25%,50%,75%,max
rv,4511.0,0.162,0.117,0.033,0.091,0.132,0.195,0.979
atrp,4518.0,0.013,0.008,0.004,0.008,0.011,0.016,0.071
er,4511.0,0.233,0.163,0.000,0.102,0.209,0.331,0.828
adx,4505.0,22.643,7.579,8.137,16.920,21.523,27.354,55.046
vr,4407.0,0.841,0.144,0.329,0.738,0.849,0.942,1.485


**Efficiency Ratio, intuition.** Two paths with the same start and end: one straight, one zigzag.

In [10]:
rng = np.random.default_rng(3)
n = 21
straight = np.linspace(0, 10, n) + rng.normal(0, 0.15, n)
zigzag = np.linspace(0, 10, n) + np.where(np.arange(n) % 2, 1, -1) * rng.uniform(2, 4, n)
zigzag[0], zigzag[-1] = 0, 10
fig = go.Figure()
for name, p in [("straight", straight), ("zigzag", zigzag)]:
    er = abs(p[-1] - p[0]) / np.abs(np.diff(p)).sum()
    fig.add_trace(go.Scatter(y=p, mode="lines+markers", name=f"{name} (ER = {er:.2f})"))
fig.update_layout(title="Same start, same end, very different efficiency", xaxis_title="bar", height=400)
fig.show()

### Relative thresholds: rolling percentiles

An ATR of 0.0015 or a realized volatility of 12% means nothing across different years or instruments. Every indicator is therefore converted into its **rolling percentile rank**: where the current value stands among the last 252 values (0 = lowest, 1 = highest). "High volatility" becomes "volatility in the top 30% of the last year", which adapts to the market.

Look-ahead rule used everywhere in the notebook: the regime at time *t* only uses data up to *t − 1* (`shift(1)`), i.e. the last **closed** bar. The first `PCT_WINDOW` bars are warm-up and are not used.

In [11]:
def pct_rank(x, w=PCT_WINDOW):
    return x.rolling(w, min_periods=w).rank(pct=True)

pr = ind.apply(pct_rank)            # rolling percentile rank of each indicator
feat = pr.shift(1)                  # what is known when bar t opens: data up to t-1
T0 = feat.dropna().index[0]         # end of the warm-up
print(f"first usable bar: {T0.date()}")

first usable bar: 2006-06-30


In [12]:
# volatility: rv and its moving 70th percentile
thr_vol = ind.rv.rolling(PCT_WINDOW).quantile(VOL_Q)
high_vol_plain = (ind.rv > thr_vol)

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, row_heights=[0.4, 0.3, 0.3], vertical_spacing=0.04,
                    subplot_titles=(f"{TICKER} (log)", "Realized volatility and its rolling 70th percentile", "Efficiency ratio and its rolling 55th percentile"))
fig.add_trace(go.Scatter(x=prices.index, y=prices.Close, name="close", line_width=1.5), row=1, col=1)
fig.add_trace(go.Scatter(x=ind.index, y=ind.rv, name="rv", line_width=1.5), row=2, col=1)
fig.add_trace(go.Scatter(x=ind.index, y=thr_vol, name="rolling 70th pct", line=dict(color="orange", width=2)), row=2, col=1)
fig.add_trace(go.Scatter(x=ind.index, y=ind.er, name="ER", line_width=1), row=3, col=1)
fig.add_trace(go.Scatter(x=ind.index, y=ind.er.rolling(PCT_WINDOW).quantile(DIR_Q), name="rolling 55th pct",
                         line=dict(color="orange", width=2)), row=3, col=1)
shade(fig, high_vol_plain.loc[T0:], "red", row=2, col=1)
fig.update_yaxes(type="log", row=1, col=1)
fig.update_layout(height=800, hovermode="x unified", title="Relative thresholds adapt to the volatility level of each period")
fig.show()

Many indicators measure similar things. The correlation of their percentile ranks shows which ones are redundant (this matters later, when we look at which indicator "explains" a result).

In [13]:
corr = pr.loc[T0:].corr(method="spearman")
fig = go.Figure(go.Heatmap(z=corr.values, x=corr.columns, y=corr.index, zmin=-1, zmax=1, colorscale="RdBu", text=corr.round(2).values,
                           texttemplate="%{text}"))
fig.update_layout(title="Spearman correlation of the percentile ranks", height=450, yaxis_autorange="reversed")
fig.show()

`rv` and `atrp` are almost the same measure (Spearman 0.90), so they will tell the same story. The efficiency ratio is moderately and negatively related to volatility (-0.31 with `rv`): when the market is turbulent, price moves less efficiently. `adx` and `er` are related (0.30), while the variance ratio is practically uncorrelated with everything else (about 0).

## 4. Classification: rule-based and statistical

### 4.1 Rule-based: the 2×2 matrix

Thresholds on the two main indicators (`rv` for volatility, `er` for directionality), both as rolling percentiles:

- high volatility when the percentile of `rv` is above `VOL_Q` (70%);
- trending when the percentile of `er` is above `DIR_Q` (55%).

To avoid a regime that flips every few bars we use **hysteresis** (the exit threshold is `BAND` away from the entry one) and a **minimum duration** (a change is confirmed only after `MIN_DUR` consecutive bars). Section 7 shows why. The function below turns a score in [0, 1] into a causal on/off state.

In [14]:
def hysteresis(f, q_off, band=BAND, min_dur=MIN_DUR):
    """Causal on/off state of a score f: switches off when f < q_off, back on when f > q_off + band.
    A switch is confirmed only after min_dur consecutive bars in the new raw state (NaN while f is NaN)."""
    v, out = f.to_numpy(), np.full(len(f), np.nan)
    raw_on = conf = None
    run = 0
    for i, x in enumerate(v):
        if np.isnan(x):
            continue
        if raw_on is None:
            raw_on = conf = x > q_off
        elif raw_on and x < q_off:
            raw_on = False
        elif not raw_on and x > q_off + band:
            raw_on = True
        run = run + 1 if raw_on != conf else 0
        if run >= min_dur:
            conf, run = raw_on, 0
        out[i] = conf
    return pd.Series(out, f.index)

low_vol = hysteresis(1 - feat.rv, 1 - VOL_Q).astype(bool)          # on while percentile(rv) < VOL_Q
trending = hysteresis(feat.er, DIR_Q - BAND).astype(bool)          # on while percentile(er) > DIR_Q
valid = feat.notna().all(axis=1)

QUAD = {(True, True): "Quiet trend", (False, True): "Strong trend", (True, False): "Calm range", (False, False): "Choppy / crisis"}
QCOLOR = {"Quiet trend": "green", "Strong trend": "dodgerblue", "Calm range": "grey", "Choppy / crisis": "red"}
regime = pd.Series([QUAD[(lv, tr)] for lv, tr in zip(low_vol, trending)], prices.index).where(valid)
print(regime.value_counts(normalize=True).round(3).to_string())

Quiet trend        0.349
Calm range         0.280
Choppy / crisis    0.243
Strong trend       0.128


In [15]:
fig = go.Figure(go.Scatter(x=prices.index, y=prices.Close, name="close", line=dict(color="white", width=1.5)))
for q, color in QCOLOR.items():
    shade(fig, regime == q, color, opacity=0.3)
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", marker=dict(color=color, size=12, symbol="square"), name=q))
fig.update_yaxes(type="log")
fig.update_layout(title="Rule-based regimes: volatility × directionality", height=550)
fig.show()

Over the usable sample the market spends 35% of the time in the quiet trend, 28% in the calm range, 24% in the choppy/crisis regime and only 13% in the strong trend: high volatility goes mostly together with low efficiency (the negative correlation we saw above), so strong trends in turbulent markets are rare.

### 4.2 Statistical: Hidden Markov Model

A Gaussian HMM assumes the market is in one of a few hidden states, each with its own mean and variance of returns, and that the state changes following a transition matrix. We use **2 states** (volatility dimension) fitted on the daily returns. Two precautions:

1. The parameters are estimated **in-sample only** (up to `IS_END`).
2. States are sorted by variance (state 0 = low volatility, state 1 = high volatility) to avoid the arbitrary label order of the fit.

`hmmlearn` returns `predict_proba`, which are **smoothed** probabilities: they use the whole series, future included. For trading we need **filtered** probabilities, P(state at *t* | data up to *t*), so we compute them with a forward pass written from scratch and check it against `hmmlearn` on the last bar, where filtered and smoothed coincide.

In [16]:
X_all = (logret * 100).to_numpy()[:, None]                                 # returns in %
X_is = (logret.loc[:IS_END] * 100).to_numpy()[:, None]

fits = [GaussianHMM(2, covariance_type="full", n_iter=500, random_state=s).fit(X_is) for s in range(10)]
hmm = max(fits, key=lambda m: m.score(X_is))                                # best of 10 random starts
order = np.argsort(hmm.covars_.ravel())                                     # state 0 = low variance
mu, var = hmm.means_.ravel()[order], hmm.covars_.ravel()[order]
trans = hmm.transmat_[np.ix_(order, order)]
start = hmm.startprob_[order]

def hmm_filtered(x, mu, var, trans, start):
    """Forward algorithm (scaled): P(state_t | x_1..x_t), no future data."""
    b = np.column_stack([stats.norm.pdf(x, m, np.sqrt(v)) for m, v in zip(mu, var)])
    alpha = np.zeros_like(b)
    a = start * b[0]
    alpha[0] = a / a.sum()
    for t in range(1, len(x)):
        a = (alpha[t - 1] @ trans) * b[t]
        alpha[t] = a / a.sum()
    return alpha

p_filt = pd.DataFrame(hmm_filtered(X_all[:, 0], mu, var, trans, start), logret.index, ["low_vol", "high_vol"])
p_smooth = pd.DataFrame(hmm.predict_proba(X_all)[:, order], logret.index, ["low_vol", "high_vol"])

# check on the last bar: the two must coincide there
print("filtered last bar:", p_filt.iloc[-1].round(6).to_numpy(), "| smoothed (hmmlearn) last bar:", p_smooth.iloc[-1].round(6).to_numpy())
print("max |smoothed - filtered| over the sample:", (p_smooth - p_filt).abs().max().max().round(3))

filtered last bar: [0.490095 0.509905] | smoothed (hmmlearn) last bar: [0.490095 0.509905]
max |smoothed - filtered| over the sample: 0.839


In [17]:
print("HMM states (fitted in-sample):")
display(pd.DataFrame({"daily mean %": mu, "daily std %": np.sqrt(var), "annualized vol %": np.sqrt(var) * np.sqrt(BARS_PER_YEAR),
                      "P(stay)": np.diag(trans), "expected duration (days)": 1 / (1 - np.diag(trans))}, index=["low_vol", "high_vol"]).round(3))

# is the high-vol state readable or driven by a few outliers? Share of high-vol days by year (filtered > 0.5)
hv = (p_filt.high_vol > 0.5)
by_year = hv.groupby(hv.index.year).mean()
fig = go.Figure(go.Bar(x=by_year.index, y=by_year.values))
fig.update_layout(title="Share of days in the high-volatility HMM state, by year", yaxis_tickformat=".0%", height=400)
fig.show()
print(f"overall share of high-vol days: {hv.mean():.1%}; years with more than 30% high-vol days: {list(by_year.index[by_year > 0.3])}")

HMM states (fitted in-sample):


,daily mean %,daily std %,annualized vol %,P(stay),expected duration (days)
low_vol,0.072,0.734,11.658,0.993,134.611
high_vol,-0.139,2.274,36.098,0.977,42.631


overall share of high-vol days: 23.4%; years with more than 30% high-vol days: [2008, 2009, 2011, 2020, 2022]


In [18]:
# filtered state probability under the price, next to the rule-based volatility state
ph = p_filt.high_vol.shift(1)                       # known at the open of bar t: data up to t-1
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, row_heights=[0.4, 0.3, 0.3], vertical_spacing=0.04,
                    subplot_titles=(f"{TICKER} (log)", "HMM filtered P(high-vol state), shifted by 1 bar", "Rule-based high-vol state"))
fig.add_trace(go.Scatter(x=prices.index, y=prices.Close, showlegend=False, line_width=1.5), row=1, col=1)
fig.add_trace(go.Scatter(x=ph.index, y=ph, showlegend=False, line_width=1.5), row=2, col=1)
fig.add_trace(go.Scatter(x=low_vol.index, y=(~low_vol & valid).astype(int), showlegend=False, line_width=1.5, line_shape="hv"), row=3, col=1)
fig.update_yaxes(type="log", row=1, col=1)
fig.update_layout(height=750, hovermode="x unified")
fig.show()

hmm_hv = (ph > 0.5).loc[T0:]
rule_hv = (~low_vol).loc[T0:]
print(f"HMM and rule-based volatility regime agree on {(hmm_hv == rule_hv).mean():.1%} of the days "
      f"(HMM high-vol {hmm_hv.mean():.1%}, rule-based high-vol {rule_hv.mean():.1%})")

HMM and rule-based volatility regime agree on 75.9% of the days (HMM high-vol 25.5%, rule-based high-vol 37.1%)


The two HMM states are readable as volatility regimes: a low-volatility state (about 11.7% annualized, positive mean return, expected duration of about 135 days) and a high-volatility state (about 36.1% annualized, negative mean return of -0.14% per day, expected duration of about 43 days). The high-volatility state is not driven by a single episode: the filtered probability is above 0.5 on 63% of the days of 2008–2009 and 47% of 2020, but also on 70% of 2022 and 39% of 2011, while it is above 0.5 on only 17% of the days outside 2008–2009 and 2020. The rule-based volatility regime agrees with the HMM on 76% of the days; it flags more days as high volatility (37.1% against 25.5%) because a percentile threshold labels about 30% of the days by construction and hysteresis keeps the state on for longer.

## 5. Trap #1: look-ahead bias in classification

The most common way to get a beautiful and useless regime filter is to use information that was not available at the time. We compare the **wrong** way with the **right** way on the same filter, "do not enter new trades while volatility is high":

| | wrong (look-ahead) | right |
|---|---|---|
| threshold | 70th percentile of the **whole sample** | **rolling** 70th percentile |
| HMM | **smoothed** probabilities (use the future) | **filtered** probabilities (use the past) |

In both cases the regime of bar *t* is read from data up to *t − 1*: the only difference is the future information hidden in the threshold or in the smoothing. A trade is allowed only if the filter is "on" at its entry bar.

First the plumbing that applies a filter to the trades of a strategy. The base trades are always generated by the unfiltered strategy; the filter only decides which of them are taken (a simplification: a skipped trade does not free the strategy to take a different signal).

In [19]:
def prepare(strats, feat):
    """Trades with the regime features known at entry, and per-bar returns from the first usable bar on."""
    t0 = feat.dropna().index[0]
    out = {}
    for name, (tr, ret, tid) in strats.items():
        tr = tr[tr.entry_date >= t0]
        bad = ~tid.isin(tr.index)
        ret, tid = ret.mask(bad, 0.0), tid.mask(bad, -1)
        tr = tr.join(feat.loc[tr.entry_date].set_axis(tr.index))
        out[name] = dict(trades=tr, ret=ret.loc[t0:], tid=tid.loc[t0:])
    return out

def trade_keep(S, state):
    """True for the trades entered while the filter state is on (state: boolean Series indexed by bar)."""
    return state.reindex(S["trades"].entry_date).fillna(True).to_numpy(dtype=bool)

def kept_bars(S, state):
    """True on the bars of the trades allowed by the filter."""
    tid = S["tid"].to_numpy()
    lookup = np.zeros(tid.max() + 2, bool)           # last slot (index -1) = no trade open = False
    lookup[S["trades"].index] = trade_keep(S, state)
    return pd.Series(lookup[tid], S["tid"].index)

def filtered(S, state):
    """Per-bar returns of the strategy keeping only the trades allowed by the filter."""
    return S["ret"] * kept_bars(S, state)

def in_sample(tr):
    return tr[tr.entry_date <= IS_END]

S = prepare(raw_strats, feat)
IS = slice(None, IS_END)
print({k: len(v["trades"]) for k, v in S.items()}, "trades from", S["trend"]["ret"].index[0].date())

{'trend': 70, 'meanrev': 205} trades from 2006-06-30


In [20]:
rv_prev = ind.rv.shift(1)
thr_full = ind.rv.loc[T0:].quantile(VOL_Q)
states = {
    "full-sample threshold (look-ahead)": rv_prev <= thr_full,
    "rolling threshold": feat.rv <= VOL_Q,
    "HMM smoothed (look-ahead)": p_smooth.high_vol.shift(1) <= 0.5,
    "HMM filtered": p_filt.high_vol.shift(1) <= 0.5,
}
colors = {"unfiltered": "grey", "full-sample threshold (look-ahead)": "red", "rolling threshold": "limegreen",
          "HMM smoothed (look-ahead)": "orange", "HMM filtered": "dodgerblue"}

fig = make_subplots(rows=1, cols=2, subplot_titles=("Trend follower", "Mean reversion"), shared_yaxes=True)
table = {}
for col, name in enumerate(S, 1):
    variants = {"unfiltered": S[name]["ret"]} | {k: filtered(S[name], st) for k, st in states.items()}
    for k, r in variants.items():
        fig.add_trace(go.Scatter(x=r.index, y=equity(r), name=k, legendgroup=k, showlegend=col == 1,
                                 line=dict(color=colors[k], width=2)), row=1, col=col)
        table[(name, k)] = dict(sharpe=sharpe(r), max_dd=max_dd(r), total_return=equity(r).iloc[-1] - 1)
fig.update_layout(height=550, title="Block trades in high volatility: same filter, with and without future information", hovermode="x unified")
fig.show()
display(pd.DataFrame(table).T.round(2))

sharpe  max_dd  total_return
trend   unfiltered                            0.53   -0.27          1.16
        full-sample threshold (look-ahead)    0.45   -0.11          0.51
        rolling threshold                     0.69   -0.09          1.01
        HMM smoothed (look-ahead)             0.57   -0.11          0.84
        HMM filtered                          0.59   -0.14          0.79
meanrev unfiltered                            0.32   -0.30          0.65
        full-sample threshold (look-ahead)    0.40   -0.14          0.38
        rolling threshold                     0.70   -0.08          0.81
        HMM smoothed (look-ahead)             0.77   -0.09          0.67
        HMM filtered                          0.25   -0.15          0.19

The Sharpe ratios are in the table: the trend follower goes from 0.53 (unfiltered) to 0.69 with the rolling threshold, and the mean reversion from 0.32 to 0.70.

Look-ahead does not always flatter the result, and that is part of the danger: you cannot tell how much it distorts. A constant threshold computed on the whole sample is slightly *worse* than the correct rolling one in both strategies here (0.45 vs 0.69, 0.40 vs 0.70). The effect of smoothing is the opposite and large for mean reversion: with the same HMM, the smoothed probabilities give a Sharpe of 0.77, the filtered ones only 0.25, which is *worse* than no filter at all (0.32). For the trend follower the two are close (0.57 vs 0.59). The smoothed probabilities look like a great filter only because they know about the future.

The same comparison on the HMM probabilities themselves, around the 2008 crisis: smoothed (uses all the data) and filtered (uses the past only).

In [21]:
z = slice("2007-06-01", "2009-12-31")
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.5, 0.5], vertical_spacing=0.05,
                    subplot_titles=(f"{TICKER}", "P(high-vol state): smoothed vs filtered"))
fig.add_trace(go.Scatter(x=prices.loc[z].index, y=prices.Close.loc[z], showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=p_smooth.loc[z].index, y=p_smooth.high_vol.loc[z], name="smoothed (uses the future)", line_color="orange"), row=2, col=1)
fig.add_trace(go.Scatter(x=p_filt.loc[z].index, y=p_filt.high_vol.loc[z], name="filtered (past only)", line_color="dodgerblue"), row=2, col=1)
fig.update_layout(height=600, hovermode="x unified")
fig.show()

# delay of the filtered state at the start of each high-vol spell of the smoothed state (search window: 60 bars)
sm_hv, fl_hv = p_smooth.high_vol > 0.5, p_filt.high_vol > 0.5
delays = []
for a, _ in runs(sm_hv):
    w = fl_hv.iloc[sm_hv.index.get_loc(a):][:60]
    delays.append(int(np.argmax(w.to_numpy())) if w.any() else np.nan)
delays = pd.Series(delays)
print(f"high-vol spells (smoothed): {len(delays)}; filtered state starts later in {(delays > 0).mean():.0%} of them, "
      f"median delay {delays.median():.0f} bars, mean {delays.mean():.1f} bars")
print(f"days where the two states disagree: {(sm_hv != fl_hv).mean():.1%}")

high-vol spells (smoothed): 30; filtered state starts later in 80% of them, median delay 2 bars, mean 1.9 bars
days where the two states disagree: 6.9%


Of the 30 high-volatility spells identified by the smoothed probabilities, the filtered ones enter the same state later in 80% of the cases (median delay of 2 bars) and the two states disagree on 6.9% of the days. A delay of a couple of days is invisible on a long chart, but it is the whole difference for a strategy whose trades last 4 bars on average.

The third form of look-ahead, indicators computed on the bar that has not closed yet, cannot be seen in a backtest on closed bars: it works in the backtest and repaints in live trading. The rule is built into the notebook: every feature is shifted by one bar, so the regime used at bar *t* comes from the last **closed** bar.

## 6. Persistence: transition matrix and regime duration

A filter is only useful if regimes last long enough to be recognised and used. The **transition matrix** gives the probability to be in each regime tomorrow given the regime today (rule-based 2×2 regimes), and the **duration** is the number of consecutive bars spent in a regime.

In [22]:
reg = regime.dropna()
labels = list(QCOLOR)
trans_emp = pd.crosstab(reg.shift(1).dropna(), reg.iloc[1:], normalize="index").reindex(index=labels, columns=labels).fillna(0)

spell_id = (reg != reg.shift(1)).cumsum()
spells = reg.groupby(spell_id).agg(["first", "size"])
spells.columns = ["regime", "bars"]

fig = make_subplots(rows=1, cols=2, subplot_titles=("Daily transition matrix (row: today, column: tomorrow)", "Duration of the regime spells (bars)"),
                    column_widths=[0.45, 0.55])
fig.add_trace(go.Heatmap(z=trans_emp.values, x=labels, y=labels, text=trans_emp.round(3).values, texttemplate="%{text}",
                         colorscale="Viridis", showscale=False), row=1, col=1)
for q in labels:
    fig.add_trace(go.Histogram(x=spells.bars[spells.regime == q], name=q, marker_color=QCOLOR[q], xbins=dict(start=0, end=150, size=5),
                               opacity=0.7), row=1, col=2)
fig.update_yaxes(autorange="reversed", row=1, col=1)
fig.update_xaxes(title_text="bars", row=1, col=2)
fig.update_layout(height=450, barmode="overlay")
fig.show()

dur = spells.groupby("regime").bars.agg(spells="count", mean_bars="mean", median_bars="median", max_bars="max").reindex(labels)
display(dur.round(1))

,spells,mean_bars,median_bars,max_bars
regime,,,,
Quiet trend,69,21.0,17.0,94
Strong trend,45,11.8,8.0,45
Calm range,77,15.1,11.0,64
Choppy / crisis,54,18.7,16.5,82


The daily transition matrix has a strong diagonal (from 0.92 for the strong trend to 0.95 for the quiet trend): it is likely to stay in the same regime tomorrow. But at the scale of spells this means short lives: the average duration is about 21 bars for the quiet trend, 19 for the choppy/crisis regime, 15 for the calm range and only 12 for the strong trend (median 8).

**Duration against the lag of the indicators.** An indicator needs time to notice a regime change. As a rough estimate, the lag of a smoothed indicator is the centre of mass of its window (half the window for a rolling mean, `n − 1` for a Wilder average), plus 1 bar for the `shift(1)`, plus the `MIN_DUR − 1` bars of confirmation. For each indicator we compute the same on/off state used for the directional regime (percentile above `DIR_Q`, hysteresis, minimum duration) and compare the length of its spells with its lag.

In [23]:
LAG_CORE = {"rv": 9.5, "atrp": ATR_N - 1, "er": 10, "adx": 2 * (ATR_N - 1), "vr": 60}      # centre of mass of the window, in bars
rows = {}
for c in pr.columns:
    st = hysteresis(feat[c], DIR_Q - BAND).dropna().astype(bool)
    sp = st.groupby((st != st.shift(1)).cumsum()).size()
    lag = LAG_CORE[c] + 1 + MIN_DUR - 1
    rows[c] = dict(est_lag_bars=lag, spells=len(sp), mean_duration=sp.mean(), median_duration=sp.median(),
                   spells_shorter_than_lag=(sp < lag).mean())
lag_table = pd.DataFrame(rows).T
display(lag_table.round(2))

lag_regime = {"rv": 9.5 + MIN_DUR, "er": 10 + MIN_DUR}
print("2x2 regime spells shorter than the combined rv/er lag (~%d bars): %.0f%%" % (max(lag_regime.values()), 100 * (spells.bars < max(lag_regime.values())).mean()))

,est_lag_bars,spells,mean_duration,median_duration,spells_shorter_than_lag
rv,14.5,90.0,47.32,24.5,0.21
atrp,18.0,71.0,60.08,36.0,0.27
er,15.0,194.0,21.95,18.0,0.42
adx,31.0,135.0,31.50,25.0,0.62
vr,65.0,60.0,69.25,41.0,0.60


2x2 regime spells shorter than the combined rv/er lag (~15 bars): 56%


The volatility indicators are fast compared with the regimes they detect: the estimated lag is 14–18 bars and the mean duration of their states 47–60 bars, with 21–27% of the spells shorter than the lag. The efficiency ratio is more problematic: lag of about 15 bars against a mean duration of 22 (median 18) and 42% of the spells shorter than the lag. `adx` and `vr` have longer lags (31 and 65 bars) and more than half of their spells (62% and 60%) end before the indicator has had the time to see them. For the combined 2×2 regime, 56% of the spells are shorter than about 15 bars. Directional regimes are the hard part: in a good fraction of the cases the regime is over by the time we know we are in it.

## 7. Trap #2: whipsaw at the thresholds

When an indicator hovers around its threshold the regime flips every few bars, and a strategy that follows it turns on and off continuously. Two fixes, both already used above: **hysteresis** (a higher threshold to enter a regime than to leave it) and a **minimum duration** (the new regime must be confirmed for a few bars). Number of regime changes with the three versions, on the same data:

In [24]:
def n_switches(st):
    st = st.dropna().astype(bool)
    return int((st != st.shift(1)).sum() - 1)

variants = {
    "single threshold": dict(band=0.0, min_dur=1),
    "+ hysteresis": dict(band=BAND, min_dur=1),
    "+ hysteresis + minimum duration": dict(band=BAND, min_dur=MIN_DUR),
}
rows = {}
for label, kw in variants.items():
    v = hysteresis(1 - feat.rv, 1 - VOL_Q, **kw)                                  # low-vol state: entry of high vol at VOL_Q
    d = hysteresis(feat.er, DIR_Q if kw["band"] == 0 else DIR_Q - BAND, **kw)    # trending state: entry at DIR_Q
    rows[label] = dict(vol_switches=n_switches(v), dir_switches=n_switches(d),
                       vol_mean_duration=len(v.dropna()) / (n_switches(v) + 1), dir_mean_duration=len(d.dropna()) / (n_switches(d) + 1))
display(pd.DataFrame(rows).T.round(1))

z = slice("2012-01-01", "2013-06-30")
st_plain = hysteresis(feat.er, DIR_Q, band=0.0, min_dur=1)
st_hys = hysteresis(feat.er, DIR_Q - BAND, band=BAND, min_dur=MIN_DUR)
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.6, 0.4], vertical_spacing=0.05,
                    subplot_titles=("Percentile of the efficiency ratio", "Trending state"))
fig.add_trace(go.Scatter(x=feat.loc[z].index, y=feat.er.loc[z], name="ER percentile", line_width=1.5), row=1, col=1)
fig.add_hline(y=DIR_Q, line_dash="dash", line_color="orange", row=1, col=1)
fig.add_hline(y=DIR_Q - BAND, line_dash="dot", line_color="orange", row=1, col=1)
fig.add_trace(go.Scatter(x=st_plain.loc[z].index, y=st_plain.loc[z] + 0.03, name="single threshold", line_shape="hv", line_color="red"), row=2, col=1)
fig.add_trace(go.Scatter(x=st_hys.loc[z].index, y=st_hys.loc[z] - 0.03, name="hysteresis + min. duration", line_shape="hv", line_color="limegreen"), row=2, col=1)
fig.update_layout(height=600, hovermode="x unified")
fig.show()

,vol_switches,dir_switches,vol_mean_duration,dir_mean_duration
single threshold,161.0,686.0,26.3,6.2
+ hysteresis,95.0,562.0,44.4,7.6
+ hysteresis + minimum duration,69.0,193.0,60.8,22.0


Over the usable sample the volatility state changes 161 times with a single threshold, 95 with hysteresis and 69 with hysteresis plus a minimum duration (mean duration from 26 to 61 bars). The directional state is much noisier: 686 changes with a single threshold (a new regime every 6 bars on average), 562 with hysteresis only and 193 with the minimum duration too (22 bars on average). The hysteresis band alone is not enough for the efficiency ratio; the minimum duration does most of the work. The price is a further lag of `MIN_DUR` bars.

## 8. Strategy performance in the four quadrants

From here on we work on the **in-sample** period only (until `IS_END`): the analysis of where the strategies work is the part that "fits" something, so it must not see the out-of-sample years. Each trade is assigned to the regime active at its entry bar (known from the previous close), and its result is the R-multiple.

In [25]:
def quad_table(name, boot=2000, seed=0):
    tr = in_sample(S[name]["trades"]).copy()
    tr["regime"] = regime.reindex(tr.entry_date).to_numpy()
    g = tr.groupby("regime").R
    out = pd.DataFrame({"trades": g.size(), "win_rate": g.apply(lambda x: (x > 0).mean()), "mean_R": g.mean(), "std_err": g.sem()})
    out = out.reindex(labels)
    out["trades"] = out["trades"].fillna(0)
    return out

quads = {name: quad_table(name) for name in S}
fig = make_subplots(rows=1, cols=2, subplot_titles=("Trend follower", "Mean reversion"), shared_yaxes=True)
for col, (name, t) in enumerate(quads.items(), 1):
    fig.add_trace(go.Bar(x=labels, y=t.mean_R, error_y=dict(type="data", array=1.96 * t.std_err), marker_color=[QCOLOR[q] for q in labels],
                         text=[f"n={int(n)}" for n in t.trades], textposition="outside", showlegend=False), row=1, col=col)
fig.update_layout(height=500, yaxis_title="mean R per trade (bars: ±1.96 s.e.)", title="In-sample expectancy by regime at entry")
fig.show()
for name, t in quads.items():
    print(name); display(t.round(3))

trend


,trades,win_rate,mean_R,std_err
regime,,,,
Quiet trend,4,0.500,-0.074,0.412
Strong trend,2,1.000,1.785,0.739
Calm range,11,0.636,0.639,0.287
Choppy / crisis,18,0.333,-0.109,0.182


meanrev


,trades,win_rate,mean_R,std_err
regime,,,,
Quiet trend,39,0.692,0.164,0.077
Strong trend,10,0.800,0.143,0.157
Calm range,27,0.593,-0.014,0.102
Choppy / crisis,26,0.654,0.009,0.099


The numbers do not follow the textbook labels, and that is useful to know before building a filter on them. The **trend follower** earns most in the *calm range* (11 trades, +0.64R, standard error 0.29): its breakouts start from quiet ranges, where the efficiency ratio is still low. It loses in the choppy/crisis regime (18 trades, -0.11R), and the quiet trend and strong trend cells have only 4 and 2 trades, too few to say anything. The **mean reversion** earns most in the *quiet trend* (39 trades, +0.16R): it buys the dips of an orderly uptrend, and it does nothing in the calm range (27 trades, -0.01R) or in the choppy regime (26 trades, +0.01R). Note the sample: only 35 trend follower trades in-sample, so the intervals are very wide; even the best mean reversion cell is only barely different from zero (0.16 ± 0.15).

## 9. Binary filter vs position sizing

Once we have a regime, there are two simple ways to use it: **trade / don't trade** (binary), or **scale the exposure**. We compare, in-sample, a binary filter ("no new trades in high volatility") with volatility targeting (position = target volatility / last known realized volatility, capped at 1 so there is no leverage; with the target set to the median volatility of the in-sample period).

In [26]:
TARGET_VOL = float(ind.rv.loc[T0:IS_END].median())
size = (TARGET_VOL / rv_prev).clip(upper=1.0)
print(f"target volatility (in-sample median of rv): {TARGET_VOL:.1%}")

fig = make_subplots(rows=1, cols=2, subplot_titles=("Trend follower", "Mean reversion"), shared_yaxes=True)
rows = {}
for col, name in enumerate(S, 1):
    base = S[name]["ret"].loc[:IS_END]
    on = S[name]["tid"].loc[:IS_END] >= 0
    variants = {"unfiltered": (base, on.astype(float)),
                "binary: not in high vol": (filtered(S[name], low_vol).loc[:IS_END], kept_bars(S[name], low_vol).loc[:IS_END].astype(float)),
                "sized: vol targeting": (base * size.loc[:IS_END], on * size.loc[:IS_END])}
    for k, (r, exposure) in variants.items():
        fig.add_trace(go.Scatter(x=r.index, y=equity(r), name=k, legendgroup=k, showlegend=col == 1), row=1, col=col)
        rows[(name, k)] = dict(sharpe=sharpe(r), max_dd=max_dd(r), total_return=equity(r).iloc[-1] - 1, avg_exposure=exposure.mean())
fig.update_layout(height=500, title="In-sample: binary filter vs volatility targeting", hovermode="x unified")
fig.show()
display(pd.DataFrame(rows).T.round(2))

target volatility (in-sample median of rv): 14.4%


sharpe  max_dd  total_return  avg_exposure
trend   unfiltered                 0.40   -0.27          0.34          0.55
        binary: not in high vol    0.57   -0.08          0.31          0.23
        sized: vol targeting       0.45   -0.18          0.32          0.50
meanrev unfiltered                 0.43   -0.28          0.46          0.20
        binary: not in high vol    0.46   -0.15          0.23          0.12
        sized: vol targeting       0.53   -0.13          0.31          0.16

For the trend follower the binary filter cuts the time in the market from 55% to 23% and the maximum drawdown from -27% to -8%, with almost the same total return (+31% against +34%) and a higher Sharpe (0.57 against 0.40); sizing keeps the exposure (50%) and improves less (Sharpe 0.45, drawdown -18%). For the mean reversion volatility targeting gives the best Sharpe (0.53 against 0.46 of the binary filter and 0.43 unfiltered), while both versions lower the total return (+31% and +23% against +46%). Sizing is a continuous version of the filter: it throws away less information, and its parameters (target volatility, cap) are fewer than the thresholds of the filter. Remember that this is in-sample and with few trades: the differences between the two methods are not proven.

## 10. Trade-level analysis: performance vs indicators

The 2×2 matrix cuts the market with thresholds chosen *a priori*. We can do better: let the data say where the thresholds should be. For every in-sample trade we have the value of each indicator **at entry** (as a rolling percentile, already shifted by one bar) and the outcome of the trade in **R-multiples**. Normalizing by the initial risk is essential: in a volatile market trades are simply bigger, both winners and losers, and raw P&L would show "relationships" that are just scale.

In [27]:
FEATS = ["rv", "atrp", "er", "adx", "vr"]
trades_is = {name: in_sample(S[name]["trades"]) for name in S}
display(trades_is["meanrev"][["entry_date", "exit_date", "bars", "R", "MAE", "MFE"] + FEATS].head(8).round(2))
print({k: len(v) for k, v in trades_is.items()}, "in-sample trades")

,entry_date,exit_date,bars,R,MAE,MFE,rv,atrp,er,adx,vr
19,2006-07-14,2006-07-20,5,0.41,0.43,0.47,0.95,0.89,0.18,0.40,0.80
20,2006-08-10,2006-08-16,5,0.45,0.11,0.49,0.74,0.85,0.16,0.13,0.63
21,2006-11-02,2006-11-07,4,0.40,0.22,0.48,0.02,0.10,0.52,0.94,0.81
22,2006-11-28,2006-11-30,3,0.62,0.13,0.70,0.37,0.05,0.12,0.99,0.19
23,2006-12-26,2006-12-28,3,0.53,0.01,0.59,0.44,0.09,0.21,0.50,0.23
24,2007-02-12,2007-02-14,3,0.18,0.22,0.22,0.33,0.16,0.36,0.38,0.10
25,2007-02-27,2007-02-27,1,-1.02,2.08,0.00,0.04,0.07,0.73,0.50,0.04
26,2007-02-28,2007-03-07,6,-0.11,0.62,0.44,0.90,0.74,0.40,0.57,0.00


{'trend': 35, 'meanrev': 102} in-sample trades


### 10.1 Correlation is not enough

Pearson correlation only sees *linear* relationships. If a strategy works best at medium volatility and suffers at both extremes (an inverted U), Pearson is about zero even though the relationship is real. **Spearman** (rank correlation) is more robust to outliers but is still monotonic; **mutual information** (MI) also captures non-monotonic dependence. MI is always positive in finite samples, so we compare it with the MI obtained after shuffling the R-multiples (the "noise" level).

First a synthetic illustration (labelled as such): an inverted-U relationship with noise.

In [28]:
def dep_table(x, y, n_shuffle=200, seed=0):
    """Pearson, Spearman (with p-value), mutual information and its shuffled (noise) level."""
    rng = np.random.default_rng(seed)
    mi = mutual_info_regression(x[:, None], y, n_neighbors=5, random_state=0)[0]
    mi_null = np.mean([mutual_info_regression(x[:, None], rng.permutation(y), n_neighbors=5, random_state=0)[0] for _ in range(n_shuffle)])
    sp = stats.spearmanr(x, y)
    return dict(pearson=stats.pearsonr(x, y)[0], spearman=sp[0], spearman_p=sp[1], mutual_info=mi, mi_shuffled=mi_null)

rng = np.random.default_rng(5)
x_syn = rng.uniform(0, 1, 400)
y_syn = 1.0 - 8 * (x_syn - 0.5) ** 2 + rng.normal(0, 1.2, 400)        # inverted U: works in the middle, loses at the extremes
print("synthetic inverted U:", {k: round(v, 3) for k, v in dep_table(x_syn, y_syn).items()})

synthetic inverted U: {'pearson': np.float64(-0.033), 'spearman': np.float64(-0.004), 'spearman_p': np.float64(0.939), 'mutual_info': np.float64(0.086), 'mi_shuffled': np.float64(0.012)}


In [29]:
def scatter_lowess(fig, x, y, row, col, name_x, frac=0.6, color="dodgerblue"):
    fig.add_trace(go.Scatter(x=x, y=y, mode="markers", marker=dict(size=6, opacity=0.5, color=color), showlegend=False), row=row, col=col)
    lw = lowess(y, x, frac=frac)
    fig.add_trace(go.Scatter(x=lw[:, 0], y=lw[:, 1], mode="lines", line=dict(color="orange", width=4), showlegend=False), row=row, col=col)
    fig.update_xaxes(title_text=name_x, row=row, col=col)

fig = make_subplots(rows=1, cols=3, subplot_titles=("Synthetic inverted U", "Trend follower (real, in-sample)", "Mean reversion (real, in-sample)"))
scatter_lowess(fig, x_syn, y_syn, 1, 1, "indicator")
for col, name in enumerate(S, 2):
    t = trades_is[name]
    scatter_lowess(fig, t.rv.to_numpy(), t.R.to_numpy(), 1, col, "percentile of realized vol at entry")
fig.update_yaxes(title_text="R-multiple", row=1, col=1)
fig.update_layout(height=450, title="R-multiple vs indicator, with LOWESS curve")
fig.show()

dep = {}
for name in S:
    for f in FEATS:
        dep[(name, f)] = dep_table(trades_is[name][f].to_numpy(), trades_is[name].R.to_numpy())
dep = pd.DataFrame(dep).T
display(dep.round(3))

pearson  spearman  spearman_p  mutual_info  mi_shuffled
trend   rv     -0.154    -0.194       0.264        0.094        0.024
        atrp   -0.136    -0.178       0.305        0.070        0.022
        er      0.314     0.264       0.125        0.001        0.025
        adx    -0.168    -0.175       0.316        0.000        0.025
        vr      0.331     0.334       0.050        0.093        0.027
meanrev rv     -0.131    -0.098       0.326        0.000        0.023
        atrp   -0.098    -0.055       0.580        0.082        0.021
        er      0.017     0.078       0.433        0.075        0.015
        adx     0.148     0.263       0.008        0.103        0.018
        vr      0.110     0.085       0.395        0.030        0.018

In the synthetic example the Pearson correlation is -0.03 and the Spearman -0.00, yet the mutual information (0.086) is well above its shuffled level (0.012): the relationship is real and non-linear. On the real in-sample trades the correlations are low: no |Spearman| above 0.34 and, out of 10 tests, only one clearly under a p-value of 0.05 (mean reversion with `adx`, Spearman 0.26, p = 0.008) and one at the border (trend follower with `vr`, 0.33, p = 0.050). The trend follower with `rv` (Pearson -0.15, Spearman -0.19, MI 0.094 against 0.024 of the shuffled data) and the mean reversion with `atrp` (Pearson -0.10, Spearman -0.06, MI 0.082 against 0.021) are cases where MI sees more than the linear and rank correlations. The scatter of the trend follower against volatility has a hump (see the bins below). With 35 trades, MI is noisy: `er` has Spearman 0.26 and MI of about 0. None of the real relations looks like the synthetic one, and they are all weak, as expected at single-trade level.

### 10.2 Binned analysis with confidence intervals

Split each indicator into quintiles and compute, per bin, the expectancy (mean R), the win rate and a **bootstrap confidence interval** of the mean (2000 resamples of the trades in the bin). The interval shows how uncertain each bar is: with few trades per bin it is wide.

In [30]:
def boot_ci(x, n=2000, seed=0):
    rng = np.random.default_rng(seed)
    m = rng.choice(x, (n, len(x))).mean(axis=1)
    return np.percentile(m, [2.5, 97.5])

def bin_table(t, f, q=5):
    b = pd.qcut(t[f], q, labels=False, duplicates="drop")
    rows = []
    for k, g in t.groupby(b):
        lo, hi = boot_ci(g.R.to_numpy())
        rows.append(dict(bin=k + 1, trades=len(g), win_rate=(g.R > 0).mean(), mean_R=g.R.mean(), ci_low=lo, ci_high=hi))
    return pd.DataFrame(rows).set_index("bin")

BIN_FEATS = ["rv", "er"]
fig = make_subplots(rows=2, cols=2, shared_yaxes=True, vertical_spacing=0.15,
                    subplot_titles=[f"{n} - {f}" for n in ["Trend follower", "Mean reversion"] for f in ["volatility (rv)", "efficiency ratio"]])
bins = {}
for r, name in enumerate(S, 1):
    for c, f in enumerate(BIN_FEATS, 1):
        bt = bin_table(trades_is[name], f)
        bins[(name, f)] = bt
        fig.add_trace(go.Bar(x=bt.index, y=bt.mean_R, showlegend=False, text=bt.trades.map(lambda n: f"n={n}"), textposition="outside",
                             error_y=dict(type="data", symmetric=False, array=bt.ci_high - bt.mean_R, arrayminus=bt.mean_R - bt.ci_low)), row=r, col=c)
        fig.update_xaxes(title_text="quintile of the percentile at entry (1 = lowest)", row=r, col=c)
fig.update_yaxes(title_text="mean R (95% bootstrap CI)")
fig.update_layout(height=750, title="Expectancy by quintile of the indicator, in-sample")
fig.show()
for k, v in bins.items():
    print(k); display(v.round(3))

('trend', 'rv')


,trades,win_rate,mean_R,ci_low,ci_high
bin,,,,,
1,7,0.571,-0.043,-0.524,0.444
2,7,0.571,0.787,0.129,1.476
3,7,0.857,1.143,0.571,1.703
4,7,0.143,-0.364,-0.781,0.164
5,7,0.286,-0.331,-0.704,0.007


('trend', 'er')


,trades,win_rate,mean_R,ci_low,ci_high
bin,,,,,
1,7,0.286,-0.142,-0.501,0.235
2,7,0.571,0.225,-0.346,0.841
3,7,0.286,-0.181,-0.858,0.619
4,7,0.429,0.367,-0.363,1.179
5,7,0.857,0.923,0.298,1.578


('meanrev', 'rv')


,trades,win_rate,mean_R,ci_low,ci_high
bin,,,,,
1,21,0.667,0.138,-0.070,0.315
2,20,0.650,0.105,-0.104,0.292
3,20,0.750,0.199,-0.032,0.380
4,20,0.550,-0.109,-0.332,0.089
5,21,0.714,0.040,-0.217,0.261


('meanrev', 'er')


,trades,win_rate,mean_R,ci_low,ci_high
bin,,,,,
1,21,0.571,0.015,-0.209,0.209
2,20,0.750,0.055,-0.197,0.288
3,20,0.600,0.142,0.030,0.259
4,22,0.727,0.164,-0.049,0.333
5,19,0.684,-0.010,-0.303,0.243


The trend follower against volatility has a hump: the 2nd and 3rd quintiles earn +0.79R and +1.14R (confidence intervals above zero), the 4th and 5th lose (-0.36R, -0.33R), the 1st is flat. Against the efficiency ratio only the highest quintile works (+0.92R, interval from +0.30R to +1.58R). With 7 trades per bin each of these claims rests on very little data. The mean reversion is much flatter: all the bins against volatility have intervals that include zero, and against the efficiency ratio only the 3rd quintile (+0.14R, interval +0.03R to +0.26R) is clearly positive. Wide intervals are the real message: the data tell us where the strategy *could* work, not where it *does*.

### 10.3 Two dimensions: heatmap

The continuous version of the 2×2 matrix: one indicator on each axis (volatility and efficiency ratio), expectancy as colour. With few in-sample trades we use terciles instead of quintiles, otherwise most cells would hold one or two trades. Remember that the indicators are correlated with each other (see the correlation matrix in section 3): an effect attributed to one may belong to the other.

In [31]:
def heat2d(t, fx="er", fy="rv", q=3):
    bx, by = pd.qcut(t[fx], q, labels=False, duplicates="drop"), pd.qcut(t[fy], q, labels=False, duplicates="drop")
    mean = t.groupby([by, bx]).R.mean().unstack()
    cnt = t.groupby([by, bx]).R.size().unstack()
    return mean, cnt

fig = make_subplots(rows=1, cols=2, subplot_titles=("Trend follower", "Mean reversion"), horizontal_spacing=0.12)
for col, name in enumerate(S, 1):
    mean, cnt = heat2d(trades_is[name])
    txt = mean.round(2).astype(str) + "<br>n=" + cnt.fillna(0).astype(int).astype(str)
    fig.add_trace(go.Heatmap(z=mean.values, x=["low", "mid", "high"], y=["low", "mid", "high"], text=txt.values, texttemplate="%{text}",
                             zmid=0, colorscale="RdYlGn", showscale=col == 2), row=1, col=col)
    fig.update_xaxes(title_text="efficiency ratio (percentile at entry)", row=1, col=col)
fig.update_yaxes(title_text="realized vol (percentile at entry)")
fig.update_layout(height=500, title="Mean R by volatility × efficiency ratio terciles, in-sample")
fig.show()

The cells contain between 2 and 16 trades. For the trend follower the extremes are +1.73R (mid volatility, high efficiency ratio, 4 trades) and -0.71R (high volatility, mid efficiency ratio, 5 trades); for the mean reversion all the cells are between -0.09R and +0.22R, the weakest being high volatility with a low efficiency ratio (11 trades). The picture is compatible with the previous sections (mean reversion is weaker in the high-volatility row, the trend follower is better at medium volatility), but nothing here is more than a hypothesis to validate out-of-sample.

## 11. Beyond the final result: MAE and MFE by regime

A regime may not change *whether* a trade is profitable, but *how* it gets there. **MAE** (maximum adverse excursion) is the deepest point against the position and **MFE** (maximum favorable excursion) the best point in favour, both in units of initial risk (1R = distance to the initial stop). If trades go deeper against you before working in high volatility, the answer may not be to stop trading but to adapt the stop.

In [32]:
rows, fig = {}, make_subplots(rows=1, cols=2, subplot_titles=("Trend follower", "Mean reversion"), shared_yaxes=True)
for col, name in enumerate(S, 1):
    t = trades_is[name].copy()
    t["vol_regime"] = np.where(low_vol.reindex(t.entry_date).to_numpy(), "low vol", "high vol")
    for reg_name, color in [("low vol", "limegreen"), ("high vol", "red")]:
        g = t[t.vol_regime == reg_name]
        fig.add_trace(go.Scatter(x=g.MAE, y=g.MFE, mode="markers", name=reg_name, legendgroup=reg_name, showlegend=col == 1,
                                 marker=dict(color=color, opacity=0.6, size=7)), row=1, col=col)
        rows[(name, reg_name)] = dict(trades=len(g), median_MAE=g.MAE.median(), median_MFE=g.MFE.median(), mean_R=g.R.mean(),
                                      share_stopped=(g.MAE >= 1).mean())
    fig.update_xaxes(title_text="MAE (R)", row=1, col=col)
fig.update_yaxes(title_text="MFE (R)", row=1, col=1)
fig.update_layout(height=500, title="MAE vs MFE by volatility regime at entry (in-sample)")
fig.show()
display(pd.DataFrame(rows).T.round(2))

trades  median_MAE  median_MFE  mean_R  share_stopped
trend   low vol     15.0        0.46        1.12    0.45           0.07
        high vol    20.0        0.44        0.81    0.08           0.15
meanrev low vol     66.0        0.28        0.35    0.09           0.14
        high vol    36.0        0.27        0.27    0.05           0.14

In R units the typical adverse excursion is almost identical in the two regimes (median MAE 0.46R against 0.44R for the trend follower, 0.28R against 0.27R for the mean reversion), and so is the share of trades that reach the initial stop for the mean reversion (14% in both). That is not surprising: the stop is a multiple of the ATR, so R already scales with volatility. What changes is the favourable side: the trend follower's median MFE goes from 1.12R in low volatility to 0.81R in high volatility, and the share of trades stopped from 7% to 15%. In this sample there is no evidence that the stop should be wider in high volatility; there is some evidence that the potential of the trend follower is smaller there (with 15 and 20 trades).

## 12. Trap #3: data snooping

Ten indicators × five bins × several strategies means hundreds of comparisons: something will look significant by chance. Here is the experiment on **pure noise**: 300 trades whose R-multiples are random and independent of ten random "indicators". For each of the 50 bins we test whether its mean R differs from the other trades (Welch t-test, 5% level).

In [33]:
def bin_pvalues(R, X, q=5):
    """Mean R per bin and p-value (bin vs all other trades) for every column of X."""
    mean = np.zeros((X.shape[1], q)); pval = np.ones((X.shape[1], q))
    for j in range(X.shape[1]):
        b = pd.qcut(X[:, j], q, labels=False, duplicates="drop")
        for k in range(q):
            m = b == k
            mean[j, k] = R[m].mean()
            pval[j, k] = stats.ttest_ind(R[m], R[~m], equal_var=False).pvalue
    return mean, pval

rng = np.random.default_rng(12)
R_noise, X_noise = rng.standard_t(5, 300), rng.uniform(size=(300, 10))
mean_n, p_n = bin_pvalues(R_noise, X_noise)

fig = go.Figure(go.Heatmap(z=mean_n, x=[f"bin {k}" for k in range(1, 6)], y=[f"indicator {j}" for j in range(1, 11)], zmid=0, colorscale="RdYlGn",
                           text=np.where(p_n < 0.05, "★", ""), texttemplate="%{text}", textfont_size=28))
fig.update_layout(title=f"Random data: mean R per bin. ★ = 'significant' at 5%: {(p_n < 0.05).sum()} of 50", height=550, yaxis_autorange="reversed")
fig.show()

# repeat the experiment many times
hits, hits_bonf = [], []
for s in range(500):
    r = np.random.default_rng(1000 + s)
    p = bin_pvalues(r.standard_t(5, 300), r.uniform(size=(300, 10)))[1]
    hits.append((p < 0.05).sum())
    hits_bonf.append((p < 0.05 / 50).sum())
hits, hits_bonf = np.array(hits), np.array(hits_bonf)
print(f"500 repetitions: average false positives per grid = {hits.mean():.2f}; at least one in {np.mean(hits >= 1):.0%} of the grids")
print(f"with a Bonferroni threshold (0.05/50 = 0.001): at least one false positive in {np.mean(hits_bonf >= 1):.0%} of the grids")

500 repetitions: average false positives per grid = 2.47; at least one in 90% of the grids
with a Bonferroni threshold (0.05/50 = 0.001): at least one false positive in 5% of the grids


On pure noise, a grid of 50 bins gives on average 2.5 "significant" bins, and at least one in 90% of the grids. Even a Bonferroni threshold (0.05 / 50) leaves a false positive in about 5% of the grids, as expected. This is the situation of a researcher who looks at ten indicators, five bins each, and keeps the one that "works".

The real in-sample grid of this notebook (2 strategies × 5 indicators × 5 bins = 50 tests, but with far fewer trades and correlated indicators) can be read against that baseline: how many bins pass the 5% test?

In [34]:
real_hits = []
for name in S:
    t = trades_is[name]
    mean_r, p_r = bin_pvalues(t.R.to_numpy(), t[FEATS].to_numpy())
    for j, f in enumerate(FEATS):
        for k in range(5):
            if p_r[j, k] < 0.05:
                real_hits.append(dict(strategy=name, indicator=f, bin=k + 1, mean_R=mean_r[j, k], p_value=p_r[j, k]))
print(f"significant bins at 5% on real in-sample trades: {len(real_hits)} of 50 (pure chance would give about 2.5 per 50 tests)")
if real_hits:
    display(pd.DataFrame(real_hits).round(3))

significant bins at 5% on real in-sample trades: 5 of 50 (pure chance would give about 2.5 per 50 tests)


,strategy,indicator,bin,mean_R,p_value
0,trend,rv,3,1.143,0.011
1,trend,rv,4,-0.364,0.039
2,trend,rv,5,-0.331,0.016
3,trend,atrp,5,-0.342,0.015
4,trend,vr,1,-0.506,0.001


There are 5 significant bins out of 50 tests, against about 2.5 expected from chance alone. But they are not five independent findings: three are bins of `rv`, one of `atrp` (correlated 0.90 with `rv`), all from the trend follower, which has only 35 trades, and one is the first bin of `vr`. Only that last one (p ≈ 0.001) is anywhere near a multiple-testing threshold. Every threshold we pick from this kind of analysis is a parameter, and we have just seen how easy it is to find one by chance: the next sections are the antidote.

## 13. Validation: thresholds in-sample, check out-of-sample

Everything so far describes the in-sample trades. To know whether a regime filter *works* we need to find the thresholds in-sample and verify them on data never used to choose them. Rules for this notebook:

- **Strategy parameters stay fixed.** Only the filter is optimized, otherwise we would not know where an improvement comes from.
- **The filter family is the same for every indicator.** A *favourability score* is built from the indicator's rolling percentile (the percentile itself, or `1 − percentile`, i.e. the sign of the filter), and trading is allowed while the score is above a threshold *q* (with the hysteresis and minimum duration of section 4). `q = 0` means no filter. Both the **sign** and **q** are chosen in-sample.
- **Metric: Sharpe ratio of the daily returns of the filtered strategy.** It already penalizes the time out of the market, so a filter cannot win just by trading less.
- **Plateau, not peak.** The metric of each *q* is averaged with its two neighbours and the maximum of the smoothed curve is chosen: an isolated peak is probably noise, the centre of a stable region is not.

In total there are 5 indicators × 2 strategies = 10 filters to evaluate (a number to keep in mind: section 12).

In [35]:
Q_GRID = np.round(np.arange(0.0, 0.85, 0.1), 2)           # threshold on the favourability score
SIGNS = (1, -1)                                           # +1: trade when the indicator percentile is high; -1: when it is low

STATES = {}                                               # all filter states are computed once: (indicator, sign, q) -> on/off per bar
for f in FEATS:
    for sg in SIGNS:
        score = feat[f] if sg > 0 else 1 - feat[f]
        for q in Q_GRID:
            STATES[(f, sg, q)] = hysteresis(score, q).astype(bool)

def segment_returns(Sx, state, a, b):
    """Per-bar returns in [a, b] of the trades entered in [a, b] and allowed by the filter state."""
    tr, tid = Sx["trades"], Sx["tid"].to_numpy()
    in_range = ((tr.entry_date >= a) & (tr.entry_date <= b)).to_numpy()
    lookup = np.zeros(tid.max() + 2, bool)
    lookup[tr.index] = in_range & trade_keep(Sx, state)
    return (Sx["ret"] * lookup[tid]).loc[a:b]

ALL_ON = pd.Series(True, feat.index)

def score_curves(name, f, a, b):
    """Sharpe of the filtered strategy for every (sign, q) on [a, b], raw and plateau-smoothed (mean with the 2 neighbouring q)."""
    raw = pd.DataFrame({sg: [sharpe(segment_returns(S[name], STATES[(f, sg, q)], a, b)) for q in Q_GRID] for sg in SIGNS}, index=Q_GRID)
    return raw, raw.rolling(3, center=True, min_periods=2).mean()

def select(name, f, a, b, plateau=True):
    raw, smooth = score_curves(name, f, a, b)
    m = (smooth if plateau else raw).stack()
    q, sg = m.idxmax()
    return sg, q

IS_A, IS_B = T0, pd.Timestamp(IS_END)
OOS_A, OOS_B = pd.Timestamp(IS_END) + pd.Timedelta(days=1), prices.index[-1]
print(f"in-sample {IS_A.date()} -> {IS_B.date()}, out-of-sample {OOS_A.date()} -> {OOS_B.date()}")

in-sample 2006-06-30 -> 2014-12-31, out-of-sample 2015-01-01 -> 2022-12-30


For the two main indicators (realized volatility and efficiency ratio) the curves of the metric against the threshold *q*, in-sample and out-of-sample, for the sign chosen in-sample. The vertical lines show the point chosen by the plateau rule (green) and the one the highest raw peak would have chosen (red).

In [36]:
PLOT_FEATS = ["rv", "er"]
sel = {(name, f): select(name, f, IS_A, IS_B) for f in PLOT_FEATS for name in S}
fig = make_subplots(rows=2, cols=2, shared_xaxes=True, vertical_spacing=0.12,
                    subplot_titles=[f"{n} - {f} (sign {'+' if sel[(m, f)][0] > 0 else '-'})" for f in PLOT_FEATS
                                    for m, n in [("trend", "Trend follower"), ("meanrev", "Mean reversion")]])
for r, f in enumerate(PLOT_FEATS, 1):
    for c, name in enumerate(S, 1):
        sg, q = sel[(name, f)]
        sg_pk, q_pk = select(name, f, IS_A, IS_B, plateau=False)
        raw_is, smooth_is = score_curves(name, f, IS_A, IS_B)
        raw_oos, _ = score_curves(name, f, OOS_A, OOS_B)
        leg = r == c == 1
        fig.add_trace(go.Scatter(x=Q_GRID, y=raw_is[sg], name="in-sample", line_color="dodgerblue", showlegend=leg), row=r, col=c)
        fig.add_trace(go.Scatter(x=Q_GRID, y=smooth_is[sg], name="in-sample, smoothed (plateau)", line=dict(color="dodgerblue", dash="dot"), showlegend=leg), row=r, col=c)
        fig.add_trace(go.Scatter(x=Q_GRID, y=raw_oos[sg], name="out-of-sample", line_color="orange", showlegend=leg), row=r, col=c)
        fig.add_vline(x=q, line_color="limegreen", line_width=3, row=r, col=c)
        if (sg_pk, q_pk) != (sg, q):
            fig.add_vline(x=q_pk, line_color="red", line_dash="dash", line_width=2, row=r, col=c)
fig.update_xaxes(title_text="threshold q", row=2)
fig.update_yaxes(title_text="Sharpe")
fig.update_layout(height=750, title="Filter metric vs threshold (green: plateau choice, red: raw peak choice)")
fig.show()

In [37]:
rows = {}
for name in S:
    for f in FEATS:
        sg, q = select(name, f, IS_A, IS_B)
        sg_pk, q_pk = select(name, f, IS_A, IS_B, plateau=False)
        oos = lambda s, qq: sharpe(segment_returns(S[name], STATES[(f, s, qq)], OOS_A, OOS_B))
        rows[(name, f)] = dict(sign=sg, q=q, IS_sharpe=sharpe(segment_returns(S[name], STATES[(f, sg, q)], IS_A, IS_B)),
                               OOS_sharpe=oos(sg, q), peak_sign=sg_pk, peak_q=q_pk, peak_OOS_sharpe=oos(sg_pk, q_pk),
                               unfiltered_IS=sharpe(segment_returns(S[name], ALL_ON, IS_A, IS_B)),
                               unfiltered_OOS=sharpe(segment_returns(S[name], ALL_ON, OOS_A, OOS_B)))
split = pd.DataFrame(rows).T
display(split.round(2))
print(f"mean OOS Sharpe: plateau choice {split.OOS_sharpe.astype(float).mean():.2f}, raw-peak choice {split.peak_OOS_sharpe.astype(float).mean():.2f}, "
      f"unfiltered {split.unfiltered_OOS.astype(float).mean():.2f}")

sign    q  IS_sharpe  OOS_sharpe  peak_sign  peak_q  \
trend   rv    -1.0  0.2       0.62        0.50       -1.0     0.2   
        atrp  -1.0  0.2       0.68        0.52       -1.0     0.2   
        er     1.0  0.2       0.55        0.54        1.0     0.3   
        adx   -1.0  0.7       0.77        0.20       -1.0     0.7   
        vr     1.0  0.4       0.79        0.26        1.0     0.4   
meanrev rv    -1.0  0.6       0.96        0.70       -1.0     0.6   
        atrp  -1.0  0.8       0.81        1.26       -1.0     0.7   
        er     1.0  0.7       0.68        0.09        1.0     0.8   
        adx    1.0  0.8       0.65       -0.07        1.0     0.3   
        vr     1.0  0.4       0.49       -0.14        1.0     0.3   

              peak_OOS_sharpe  unfiltered_IS  unfiltered_OOS  
trend   rv               0.50           0.40            0.72  
        atrp             0.52           0.40            0.72  
        er               0.55           0.40            0.72  
        adx              0.20           0.40            0.72  
        vr               0.26           0.40            0.72  
meanrev rv               0.70           0.43            0.20  
        atrp             0.82           0.43            0.20  
        er              -0.01           0.43            0.20  
        adx             -0.18           0.43            0.20  
        vr              -0.17           0.43            0.20

mean OOS Sharpe: plateau choice 0.39, raw-peak choice 0.32, unfiltered 0.46


Several things stand out in the table. First, almost all the filters are better in-sample than the unfiltered strategy, as they must be (we chose them there), and many degrade out-of-sample. Second, **the filters help the mean reversion more than the trend follower**: the unfiltered trend follower has an out-of-sample Sharpe of 0.72 and *every* filter is below it (0.20–0.54), while the mean reversion goes from 0.20 unfiltered to 0.70 with `rv` and 1.26 with `atrp`, both volatility filters (trade only when volatility is in the lower part of its range). The directional filters of the mean reversion do not hold (0.09, -0.07, -0.14). Third, the plateau rule is more robust than the peak: where the two choices differ (`atrp`, `er` and `adx` of the mean reversion) the plateau choice has the better out-of-sample Sharpe (1.26 against 0.82, 0.09 against -0.01, -0.07 against -0.18). On average over the ten filters the plateau choice gives an out-of-sample Sharpe of 0.39, the raw peak 0.32, and not filtering at all 0.46: a filter is not a free lunch, and the average filter does not beat doing nothing.

## 14. Anchored walk-forward on the filter, and threshold stability

A single in-sample/out-of-sample split gives one answer. A **walk-forward** repeats the procedure: the filter is re-optimized on all the data available up to each fold and then applied to the next, unseen, period; the out-of-sample segments are stitched together. Two choices specific to regime filters:

- **Anchored (expanding) in-sample window.** Regimes last months or years: a short rolling window may contain a single regime, and the filter would learn to exclude something it has never seen. The first in-sample window here goes from the end of the warm-up to `IS_END` and contains the 2008–2009 crisis, the recovery and the calm years that followed; it only grows.
- **Warm-up.** The indicators and their percentiles are computed on the whole price history, so at the start of every out-of-sample fold the lookbacks use the preceding data and are never "broken".

We also record the threshold selected in each fold: if it jumps around, the filter is probably noise, even when the stitched out-of-sample equity looks good.

In [38]:
FOLD_STARTS = ["2015-01-01", "2017-01-01", "2019-01-01", "2021-01-01"]
FOLDS = [(pd.Timestamp(a), pd.Timestamp(b) - pd.Timedelta(days=1)) for a, b in zip(FOLD_STARTS, FOLD_STARTS[1:] + [str(prices.index[-1].date() + pd.Timedelta(days=1))])]
WF_A, WF_B = FOLDS[0][0], FOLDS[-1][1]
print("folds:", [(a.date().isoformat(), b.date().isoformat()) for a, b in FOLDS])

def wfa(name, f):
    """Anchored walk-forward of the filter (strategy parameters fixed). Returns the stitched OOS state and the selections."""
    picks, parts, is_sharpes = [], [], []
    for a, b in FOLDS:
        is_b = a - pd.Timedelta(days=1)
        sg, q = select(name, f, T0, is_b)
        picks.append(dict(fold_start=a, sign=sg, q=q))
        parts.append(STATES[(f, sg, q)].loc[a:b])
        is_sharpes.append(sharpe(segment_returns(S[name], STATES[(f, sg, q)], T0, is_b)))
    state = pd.concat(parts)
    return dict(state=state, picks=pd.DataFrame(picks).set_index("fold_start"), is_sharpe=np.mean(is_sharpes))

WFA = {(name, f): wfa(name, f) for name in S for f in FEATS}

fig = make_subplots(rows=1, cols=2, subplot_titles=("Trend follower", "Mean reversion"), shared_yaxes=True)
for col, name in enumerate(S, 1):
    for f in FEATS:
        p = WFA[(name, f)]["picks"]
        fig.add_trace(go.Scatter(x=p.index, y=p.q, mode="lines+markers+text", text=np.where(p.sign > 0, "+", "−"), textposition="top center",
                                 name=f, legendgroup=f, showlegend=col == 1, line_shape="hv"), row=1, col=col)
fig.update_yaxes(title_text="selected threshold q", row=1, col=1)
fig.update_layout(height=500, title="Threshold selected in each walk-forward fold (+/− = sign of the filter)")
fig.show()
display(pd.DataFrame({k: [f"{'+' if r.sign > 0 else '-'}{r.q:.1f}" for r in v["picks"].itertuples()] for k, v in WFA.items()},
                    index=[a.date().isoformat() for a, _ in FOLDS]).T)

folds: [('2015-01-01', '2016-12-31'), ('2017-01-01', '2018-12-31'), ('2019-01-01', '2020-12-31'), ('2021-01-01', '2022-12-30')]


2015-01-01 2017-01-01 2019-01-01 2021-01-01
trend   rv         -0.2       -0.2       -0.2       -0.3
        atrp       -0.2       -0.1       -0.2       +0.0
        er         +0.2       +0.2       +0.2       -0.1
        adx        -0.7       -0.6       -0.6       -0.6
        vr         +0.4       +0.3       +0.3       +0.3
meanrev rv         -0.6       -0.6       -0.5       -0.5
        atrp       -0.8       -0.8       -0.8       -0.8
        er         +0.7       +0.7       +0.6       +0.7
        adx        +0.8       +0.8       +0.8       -0.8
        vr         +0.4       +0.4       -0.0       -0.8

Thresholds are fairly stable fold after fold for the volatility filters of the mean reversion (`rv`: q from 0.6 to 0.5, always negative sign; `atrp`: 0.8 in all four folds) and for the trend follower's `rv` (0.2, 0.2, 0.2, 0.3). The directional filters are less so: the sign of `er` for the trend follower, of `adx` for the mean reversion and of `vr` for the mean reversion changes in the last folds (for the last one, from +0.4 to -0.8). Stability does not mean usefulness, though: the `adx` filter of the trend follower selects almost the same threshold in all folds (-0.7, -0.6, -0.6, -0.6) and is among the worst out-of-sample.

## 15. The random filter benchmark and the final comparison

Any filter reduces the time in the market, and less exposure means lower drawdown and lower variance almost automatically: "the filter reduced the drawdown" proves nothing. The right benchmark is the distribution of the same metric under **random filters with the same exposure and the same average regime duration** as the real one. For each walk-forward filter we generate 1000 random on/off sequences from a two-state Markov chain (probability of leaving the "on" state = 1 / mean duration, probability of entering it set so that the long-run exposure matches), apply them to the same out-of-sample trades and compute the Sharpe ratio. A real filter must sit in the right tail of this distribution, not only above the unfiltered strategy.

In [39]:
def random_benchmark(Sx, state, a, b, n=1000, seed=0, paths=False):
    """Sharpe (and optionally equity paths) of n random Markov on/off filters matching exposure and mean on-duration of `state` on [a, b]."""
    st = state.loc[a:b].astype(bool)
    e = st.mean()
    on_spells = st.groupby((st != st.shift(1)).cumsum()).agg(["first", "size"])
    d = on_spells[on_spells["first"]]["size"].mean()
    p_off, p_on = 1 / d, e / d / (1 - e)
    rng = np.random.default_rng(seed)
    Z, z = np.empty((len(st), n), bool), rng.random(n) < e
    for t in range(len(st)):
        u = rng.random(n)
        z = np.where(z, u > p_off, u < p_on)
        Z[t] = z
    tr = Sx["trades"][(Sx["trades"].entry_date >= a) & (Sx["trades"].entry_date <= b)]
    K = np.hstack([Z[st.index.get_indexer(tr.entry_date)].T, np.zeros((n, 1), bool)])        # extra column: bars without an eligible trade
    tid = Sx["tid"].loc[a:b].to_numpy()
    local = np.full(Sx["tid"].max() + 2, len(tr))
    local[tr.index] = np.arange(len(tr))
    R = Sx["ret"].loc[a:b].to_numpy()[None, :] * K[:, local[tid]]
    sh = np.nan_to_num(R.mean(axis=1) / R.std(axis=1) * np.sqrt(BARS_PER_YEAR))     # no trade taken -> Sharpe 0
    out = dict(sharpe=sh, exposure=e, mean_duration=d)
    if paths:
        out["equity"] = np.cumprod(1 + R, axis=1)
    return out

summary, bench = {}, {}
for (name, f), w in WFA.items():
    real = segment_returns(S[name], w["state"], WF_A, WF_B)
    base = segment_returns(S[name], ALL_ON, WF_A, WF_B)
    bm = random_benchmark(S[name], w["state"], WF_A, WF_B, seed=100 * list(S).index(name) + FEATS.index(f))
    bench[(name, f)] = bm
    summary[(name, f)] = dict(family="vol" if f in VOL_IND else "dir", exposure=bm["exposure"], mean_on_duration=bm["mean_duration"],
                              unfiltered=sharpe(base), filtered=sharpe(real), random_median=np.median(bm["sharpe"]),
                              random_p95=np.percentile(bm["sharpe"], 95), percentile_vs_random=(bm["sharpe"] < sharpe(real)).mean() * 100,
                              WF_efficiency=sharpe(real) / w["is_sharpe"])
summary = pd.DataFrame(summary).T
display(summary.round(2))

family  exposure mean_on_duration unfiltered  filtered  \
trend   rv      vol  0.679742       105.307692   0.719331  0.499715   
        atrp    vol  0.829196       238.571429   0.719331  0.391372   
        er      dir  0.837637         52.71875   0.719331  0.479189   
        adx     dir  0.313307            25.24   0.719331  0.107987   
        vr      dir  0.590864             85.0   0.719331  0.259103   
meanrev rv      vol  0.360477             36.3   0.195158  0.697187   
        atrp    vol  0.182224        30.583333   0.195158  1.262361   
        er      dir  0.245283        15.935484   0.195158  0.250317   
        adx     dir  0.152433        18.058824   0.195158 -0.022266   
        vr      dir  0.553625        85.769231   0.195158   0.08225   

             random_median random_p95 percentile_vs_random WF_efficiency  
trend   rv        0.597567   0.805272                 25.8      0.853338  
        atrp      0.666497   0.791176                  2.8       0.63618  
        er         0.66349   0.819023                  8.7      0.859412  
        adx       0.410137   0.712406                 13.0      0.163644  
        vr        0.563379   0.787783                  6.1      0.403089  
meanrev rv        0.146272   0.658956                 95.7      0.813627  
        atrp      0.164516   0.698237                100.0      1.420691  
        er        0.183223   0.696342                 58.3      0.400489  
        adx       0.163056   0.668275                 25.8     -0.040075  
        vr        0.140906   0.646661                 39.5      0.193794

In [40]:
fig = make_subplots(rows=2, cols=5, subplot_titles=[f"{n} - {f}" for n in ["trend", "meanrev"] for f in FEATS], vertical_spacing=0.15)
for r, name in enumerate(S, 1):
    for c, f in enumerate(FEATS, 1):
        sh, row = bench[(name, f)]["sharpe"], summary.loc[(name, f)]
        fig.add_trace(go.Histogram(x=sh, nbinsx=40, marker_color="grey", showlegend=False), row=r, col=c)
        fig.add_vline(x=row.unfiltered, line_color="white", line_dash="dash", row=r, col=c)
        fig.add_vline(x=row.filtered, line_color="limegreen", line_width=3, row=r, col=c)
fig.update_layout(height=600, title="Out-of-sample Sharpe: 1000 random filters (grey) vs unfiltered (white dashed) vs walk-forward filter (green)")
fig.show()
by_family = summary.assign(percentile_vs_random=summary.percentile_vs_random.astype(float)).groupby("family").percentile_vs_random.agg(["mean", "median"])
display(by_family.round(1))

,mean,median
family,,
dir,25.2,19.4
vol,56.1,60.8


Read the table above column by column. **Trend follower** (unfiltered out-of-sample Sharpe 0.72): none of the five walk-forward filters beats the unfiltered strategy (0.11–0.50), and none is in the right tail of the random filters: their percentiles are between 3 and 26. Filtering reduced the exposure (31–84%) and the Sharpe with it. **Mean reversion** (0.20 unfiltered): the two volatility filters stand out, with a Sharpe of 0.70 for `rv` (95.7th percentile of the random filters, whose 95th percentile is 0.66) and 1.26 for `atrp` (above all the 1000 random filters), with a walk-forward efficiency of 0.81 and 1.42 (out-of-sample better than in-sample). The three directional filters are not distinguishable from random (percentiles 58, 26 and 40). By family, the mean percentile against the random filters is 56 for volatility and 25 for direction. The asymmetry of section 2 is therefore compatible with the data, but only because of the mean reversion; for the trend follower no filter works. Also keep in mind that 10 filters were tested: two above the 95th percentile is more than the 0.5 expected by chance, but `rv` and `atrp` are 0.90 correlated, so it is effectively one finding.

The final comparison on the stitched out-of-sample segments for the two main indicators: the unfiltered strategy, the strategy with the walk-forward filter, and the 5th–95th percentile band of the random filters.

In [41]:
fig = make_subplots(rows=2, cols=2, shared_xaxes=True, vertical_spacing=0.1,
                    subplot_titles=[f"{n} - {f}" for f in ["rv", "er"] for n in ["Trend follower", "Mean reversion"]])
for r, f in enumerate(["rv", "er"], 1):
    for c, name in enumerate(S, 1):
        w = WFA[(name, f)]
        bm = random_benchmark(S[name], w["state"], WF_A, WF_B, paths=True, seed=100 * list(S).index(name) + FEATS.index(f))
        idx = S[name]["ret"].loc[WF_A:WF_B].index
        lo, hi = np.percentile(bm["equity"], [5, 95], axis=0)
        leg = r == c == 1
        fig.add_trace(go.Scatter(x=idx, y=hi, line_width=0, showlegend=False), row=r, col=c)
        fig.add_trace(go.Scatter(x=idx, y=lo, fill="tonexty", fillcolor="rgba(150,150,150,0.35)", line_width=0, name="random filters 5-95%", showlegend=leg), row=r, col=c)
        fig.add_trace(go.Scatter(x=idx, y=equity(segment_returns(S[name], ALL_ON, WF_A, WF_B)), name="unfiltered", line_color="white", showlegend=leg), row=r, col=c)
        fig.add_trace(go.Scatter(x=idx, y=equity(segment_returns(S[name], w["state"], WF_A, WF_B)), name="walk-forward filter", line_color="limegreen", showlegend=leg), row=r, col=c)
fig.update_layout(height=750, title="Stitched out-of-sample equity (starting at 1)", hovermode="x unified")
fig.show()

## 16. The holdout: one run, at the end

The last years (from `HOLDOUT_START`) were cut away at the start and used for nothing: not for the indicators, not for the thresholds, not for the walk-forward. If we reran the walk-forward ten times trying other indicators, the walk-forward itself would become in-sample; the holdout is the one number that stays honest, **and only if it is used once**.

The procedure is fixed in advance and applies to all 10 filters, with no choice among them after seeing the result: sign and threshold are selected with the usual plateau rule on all the development data (in-sample + out-of-sample), applied to the holdout trades, and compared with 1000 random filters having the same exposure and duration. Run this cell once.

In [42]:
# features, strategies and trades recomputed on the full history (causal: nothing before the holdout changes)
ind_full = indicators(data_all)
feat_full = ind_full.apply(pct_rank).shift(1)
S_full = prepare(strategies(data_all), feat_full)
H_A, H_B = pd.Timestamp(HOLDOUT_START), data_all.index[-1]
dev_end = prices.index[-1]
ALL_ON_FULL = pd.Series(True, feat_full.index)

rows, hold_bench = {}, {}
for name in S:
    for f in FEATS:
        sg, q = select(name, f, T0, dev_end)                              # chosen on development data only
        score = feat_full[f] if sg > 0 else 1 - feat_full[f]
        state = hysteresis(score, q).astype(bool)
        real, base = segment_returns(S_full[name], state, H_A, H_B), segment_returns(S_full[name], ALL_ON_FULL, H_A, H_B)
        bm = random_benchmark(S_full[name], state, H_A, H_B, seed=100 * list(S).index(name) + FEATS.index(f))
        hold_bench[(name, f)] = bm
        rows[(name, f)] = dict(sign=sg, q=q, trades=int(((S_full[name]["trades"].entry_date >= H_A)).sum()), exposure=bm["exposure"],
                               unfiltered=sharpe(base), filtered=sharpe(real), random_median=np.median(bm["sharpe"]),
                               percentile_vs_random=(bm["sharpe"] < sharpe(real)).mean() * 100)
holdout = pd.DataFrame(rows).T
print(f"holdout: {H_A.date()} -> {H_B.date()}")
display(holdout.round(2))

holdout: 2023-01-01 -> 2026-09-30


sign    q  trades  exposure  unfiltered  filtered  \
trend   rv     1.0  0.3    18.0      0.50        1.17      0.72   
        atrp  -1.0  0.1    18.0      0.89        1.17      0.98   
        er    -1.0  0.1    18.0      0.88        1.17      1.17   
        adx   -1.0  0.7    18.0      0.34        1.17      0.46   
        vr     1.0  0.3    18.0      0.55        1.17      0.36   
meanrev rv    -1.0  0.5    52.0      0.57        0.64      0.73   
        atrp  -1.0  0.8    52.0      0.32        0.64      0.39   
        er     1.0  0.6    52.0      0.31        0.64      0.74   
        adx   -1.0  0.8    52.0      0.22        0.64     -0.03   
        vr    -1.0  0.8    52.0      0.23        0.64      0.87   

              random_median  percentile_vs_random  
trend   rv             0.83                  38.1  
        atrp           1.16                  21.5  
        er             1.15                  55.7  
        adx            0.67                  33.5  
        vr             0.88                  10.8  
meanrev rv             0.50                  74.9  
        atrp           0.41                  47.7  
        er             0.41                  77.4  
        adx            0.39                  10.8  
        vr             0.39                  85.8

The holdout does not confirm the out-of-sample picture. **Trend follower** (unfiltered Sharpe 1.17): no filter beats the unfiltered strategy (`er` is identical, 1.17, because it is almost always on) and all sit between the 11th and 56th percentile of the random filters. **Mean reversion** (unfiltered 0.64): the volatility filter that looked so good out-of-sample, `rv`, gets 0.73 (75th percentile), a modest improvement that is not distinguishable from luck, and `atrp` fails (0.39, below the unfiltered strategy, 48th percentile). The best percentiles belong to `vr` (86th) and `er` (77th), which were random-like in the walk-forward. No filter reaches the 95th percentile. With 18 and 52 trades, the percentiles are noisy, and this is a single market in a single period, but the verdict is clear: **there is no evidence that these regime filters add value out of the sample used to build them**. That is a valid result: a filter should be judged by what survives validation, not by the backtest it improves.

## Summary

- **Regimes** are described by directionality and volatility. Volatility is persistent (autocorrelation of absolute returns significant at all 40 lags) while direction is only weakly persistent.
- **Classification** needs care: rolling percentile thresholds, hysteresis and minimum duration to avoid whipsaw (directional state changes 686 times with a single threshold, 193 with the fixes), HMM probabilities **filtered**, not smoothed (same filter, Sharpe 0.77 smoothed vs 0.25 filtered for mean reversion), and indicators read on closed bars.
- **Regimes are short** compared with the lag of the indicators, especially the directional ones: 42% of the spells of the efficiency ratio and about 60% or more of those of `adx` and `vr` are shorter than the estimated lag.
- **Trade-level analysis** (R-multiples, quintile bins with bootstrap intervals, 2D heatmaps) shows where a strategy seems to work, with wide intervals; a random grid of 50 bins produces false positives in 90% of cases.
- **Validation** with plateau selection, anchored walk-forward and the random filter benchmark: in 2015–2022 the only filters clearly in the right tail of the random distribution were volatility filters of the mean reversion; none of the trend follower or directional ones was.
- **The holdout** (used once) did not confirm them. A regime filter is judged by what survives validation, not by how much it improves the backtest.

All the results are for one market, daily bars and two simple strategies: demonstrations of the method, not validations of a trading system.